###### Step 1 — Imports

In [0]:
import os
import re
import hashlib
import requests

from datetime import datetime, timezone
from urllib.parse import urljoin

from bs4 import BeautifulSoup

from delta.tables import DeltaTable

from pyspark.sql import functions as f

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

###### Step 2 — Configuration

In [0]:
bls_base_url = (
    "https://download.bls.gov/pub/time.series/pr/"
)

bls_volume_path = (
    "/Volumes/rearc/bronze/raw/bls/inbound"
)

manifest_table = (
    "rearc.audit.ingestion_manifest"
)

source_name = "bls"

user_agent = (
    "Rearc Data Quest - ankitdranjan@gmail.com"
)

##### Step 3 — Create BLS HTTP session

In [0]:
bls_session = requests.Session()

bls_session.headers.update({
    "User-Agent": user_agent
})

retry_strategy = Retry(
    total=4,
    backoff_factor=1,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504
    ],
    allowed_methods=["GET"],
    respect_retry_after_header=True
)

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

bls_session.mount(
    "https://",
    adapter
)

##### Step 4 — Helper functions

In [0]:
## 4.1 Build versioned raw path

def build_versioned_paths(
    base_path,
    source_object,
    version_dt
):
    version_stamp = version_dt.strftime(
        "%Y%m%dT%H%M%S%f"
    )

    landing_dir = (
        f"{base_path}/{source_object}"
    )

    version_dir = (
        f"{landing_dir}/v={version_stamp}"
    )

    volume_path = (
        f"{version_dir}/{source_object}"
    )

    return (
        landing_dir,
        version_dir,
        volume_path
    )

In [0]:
## 4.2 Download and calculate hash

def download_and_hash(
    session,
    source_url,
    expected_size=None
):
    response = session.get(
        source_url,
        timeout=30
    )

    response.raise_for_status()

    content = response.content

    actual_size = len(content)

    if (
        expected_size is not None
        and actual_size != expected_size
    ):
        raise ValueError(
            f"size mismatch for {source_url}. "
            f"expected={expected_size}, "
            f"actual={actual_size}"
        )

    content_hash = hashlib.sha256(
        content
    ).hexdigest()

    return (
        content,
        actual_size,
        content_hash
    )

In [0]:
## 4.3 Write immutable raw version

def write_raw_version(
    base_path,
    source_object,
    content,
    version_dt
):
    (
        landing_dir,
        version_dir,
        volume_path
    ) = build_versioned_paths(
        base_path,
        source_object,
        version_dt
    )

    os.makedirs(
        version_dir,
        exist_ok=False
    )

    with open(
        volume_path,
        "wb"
    ) as output_file:

        output_file.write(content)

    return (
        landing_dir,
        volume_path
    )

In [0]:
## 4.4 Build manifest record

def build_manifest_record(
    source_object,
    source_url,
    source_modified_dt,
    source_size,
    content_hash,
    landing_dir,
    volume_path,
    status,
    first_seen_dt,
    last_seen_dt,
    ingested_dt
):
    return {
        "source_name": source_name,
        "source_object": source_object,
        "source_url": source_url,
        "source_modified_dt": source_modified_dt,
        "source_size": source_size,
        "content_hash": content_hash,
        "landing_dir": landing_dir,
        "volume_path": volume_path,
        "status": status,
        "first_seen_dt": first_seen_dt,
        "last_seen_dt": last_seen_dt,
        "ingested_dt": ingested_dt
    }

##### Step 5 — Parse the HTML

In [0]:
## 5. Read BLS directory

response = bls_session.get(
    bls_base_url,
    timeout=30
)

response.raise_for_status()

print(
    "BLS response status:",
    response.status_code
)

soup = BeautifulSoup(
    response.text,
    "html.parser"
)

links = soup.find_all("a")

print(
    "links found:",
    len(links)
)

BLS response status: 200
links found: 13


##### Step 6 — Discover BLS Files Dynamically

In [0]:
# ---------------------------------------------------------
# Step 6 - Discover BLS source files
# ---------------------------------------------------------

directory_text = soup.get_text(
    " ",
    strip=True
)

# Pattern:
# date | time | size | filename

pattern = re.compile(
    r"(\d{1,2}/\d{1,2}/\d{4})"
    r"\s+"
    r"(\d{1,2}:\d{2}\s+[AP]M)"
    r"\s+"
    r"(\d+)"
    r"\s+"
    r"([^\s]+)"
)

matches = pattern.findall(
    directory_text
)

print(
    "Files discovered:",
    len(matches)
)

source_files = []

for (
    modified_date,
    modified_time,
    source_size,
    source_object
) in matches:

    source_modified_dt = datetime.strptime(
        f"{modified_date} {modified_time}",
        "%m/%d/%Y %I:%M %p"
    )

    source_url = urljoin(
        bls_base_url,
        source_object
    )

    source_files.append(
        {
            "source_object": source_object,
            "source_url": source_url,
            "source_modified_dt": source_modified_dt,
            "source_size": int(source_size)
        }
    )

print(
    "Total source files:",
    len(source_files)
)

Files discovered: 12
Total source files: 12


##### Step 7 — Create Source DataFrame

In [0]:
# ---------------------------------------------------------
# Step 7 - Create source DataFrame
# ---------------------------------------------------------

df_source_files = spark.createDataFrame(
    source_files
)

df_source_files = (
    df_source_files
    .select(
        "source_object",
        "source_url",
        "source_modified_dt",
        "source_size"
    )
)

display(
    df_source_files
    .orderBy(
        "source_object"
    )
)

## validation

df_source_duplicates = (
    df_source_files
    .groupBy(
        "source_object"
    )
    .count()
    .filter(
        f.col("count") > 1
    )
)

if df_source_duplicates.count() > 0:

    display(
        df_source_duplicates
    )

    raise ValueError(
        "duplicate BLS source files discovered"
    )

source_object,source_url,source_modified_dt,source_size
pr.class,https://download.bls.gov/pub/time.series/pr/pr.class,2026-09-03T08:30:00.000Z,102
pr.contacts,https://download.bls.gov/pub/time.series/pr/pr.contacts,2022-09-13T16:52:00.000Z,562
pr.data.0.Current,https://download.bls.gov/pub/time.series/pr/pr.data.0.Current,2026-09-03T08:30:00.000Z,1615931
pr.data.1.AllData,https://download.bls.gov/pub/time.series/pr/pr.data.1.AllData,2026-09-03T08:30:00.000Z,3239525
pr.duration,https://download.bls.gov/pub/time.series/pr/pr.duration,2026-09-03T08:30:00.000Z,176
pr.footnote,https://download.bls.gov/pub/time.series/pr/pr.footnote,2026-09-03T08:30:00.000Z,40
pr.measure,https://download.bls.gov/pub/time.series/pr/pr.measure,2026-09-03T08:30:00.000Z,745
pr.period,https://download.bls.gov/pub/time.series/pr/pr.period,1994-01-07T15:53:00.000Z,146
pr.seasonal,https://download.bls.gov/pub/time.series/pr/pr.seasonal,2011-11-18T16:05:00.000Z,79
pr.sector,https://download.bls.gov/pub/time.series/pr/pr.sector,2026-09-03T08:30:00.000Z,263


##### Step 8 — Read Existing BLS Manifest

In [0]:
df_manifest_bls = (
    spark.table(
        manifest_table
    )
    .filter(
        f.col("source_name")
        == source_name
    )
)

display(
    df_manifest_bls
    .orderBy(
        "source_object"
    )
)

source_name,source_object,source_url,source_modified_dt,source_size,content_hash,volume_path,status,first_seen_dt,last_seen_dt,ingested_dt,landing_dir


##### Step 9 — Read existing BLS manifest

In [0]:
df_manifest_bls = (
    df_manifest_bls
    .select(
        f.col("source_object")
            .alias("tgt_source_object"),

        f.col("source_url")
            .alias("tgt_source_url"),

        f.col("source_modified_dt")
            .alias("tgt_source_modified_dt"),

        f.col("source_size")
            .alias("tgt_source_size"),

        f.col("content_hash")
            .alias("tgt_content_hash"),

        f.col("landing_dir")
            .alias("tgt_landing_dir"),

        f.col("volume_path")
            .alias("tgt_volume_path"),

        f.col("status")
            .alias("tgt_status"),

        f.col("first_seen_dt")
            .alias("tgt_first_seen_dt"),

        f.col("last_seen_dt")
            .alias("tgt_last_seen_dt"),

        f.col("ingested_dt")
            .alias("tgt_ingested_dt")
    )
)

##### Step 10 — Full outer comparison

In [0]:
df_source_compare = (
    df_source_files
    .select(
        f.col("source_object")
            .alias("src_source_object"),

        f.col("source_url")
            .alias("src_source_url"),

        f.col("source_modified_dt")
            .alias("src_source_modified_dt"),

        f.col("source_size")
            .alias("src_source_size")
    )
)

## Join

df_compare = (
    df_source_compare.alias("src")
    .join(
        df_manifest_bls.alias("tgt"),
        f.col("src.src_source_object")
        ==
        f.col("tgt.tgt_source_object"),
        "full_outer"
    )
)

##

df_compare = (
    df_compare
    .withColumn(
        "change_type",

        f.when(
            f.col("tgt_source_object").isNull(),
            f.lit("new")
        )

        .when(
            f.col("src_source_object").isNull()
            &
            (f.col("tgt_status") == "active"),
            f.lit("removed")
        )

        .when(
            f.col("src_source_object").isNull()
            &
            (f.col("tgt_status") == "removed"),
            f.lit("already_removed")
        )

        .when(
            f.col("tgt_status") == "removed",
            f.lit("reactivated")
        )

        .when(
            ~(
                f.col(
                    "src_source_modified_dt"
                ).eqNullSafe(
                    f.col(
                        "tgt_source_modified_dt"
                    )
                )
                &
                f.col(
                    "src_source_size"
                ).eqNullSafe(
                    f.col(
                        "tgt_source_size"
                    )
                )
            ),
            f.lit(
                "changed_candidate"
            )
        )

        .otherwise(
            f.lit("unchanged")
        )
    )
)

##### Step 11 — Create a clean changes DataFrame

In [0]:
df_file_changes = (
    df_compare
    .select(
        f.coalesce(
            f.col("src_source_object"),
            f.col("tgt_source_object")
        ).alias("source_object"),

        "src_source_url",
        "src_source_modified_dt",
        "src_source_size",

        "tgt_source_url",
        "tgt_source_modified_dt",
        "tgt_source_size",
        "tgt_content_hash",
        "tgt_landing_dir",
        "tgt_volume_path",
        "tgt_status",
        "tgt_first_seen_dt",
        "tgt_last_seen_dt",
        "tgt_ingested_dt",

        "change_type"
    )
)

display(
    df_file_changes
    .orderBy(
        "source_object"
    )
)

source_object,src_source_url,src_source_modified_dt,src_source_size,tgt_source_url,tgt_source_modified_dt,tgt_source_size,tgt_content_hash,tgt_landing_dir,tgt_volume_path,tgt_status,tgt_first_seen_dt,tgt_last_seen_dt,tgt_ingested_dt,change_type
pr.class,https://download.bls.gov/pub/time.series/pr/pr.class,2026-09-03T08:30:00.000Z,102,null,null,null,null,null,null,null,null,null,null,new
pr.contacts,https://download.bls.gov/pub/time.series/pr/pr.contacts,2022-09-13T16:52:00.000Z,562,null,null,null,null,null,null,null,null,null,null,new
pr.data.0.Current,https://download.bls.gov/pub/time.series/pr/pr.data.0.Current,2026-09-03T08:30:00.000Z,1615931,null,null,null,null,null,null,null,null,null,null,new
pr.data.1.AllData,https://download.bls.gov/pub/time.series/pr/pr.data.1.AllData,2026-09-03T08:30:00.000Z,3239525,null,null,null,null,null,null,null,null,null,null,new
pr.duration,https://download.bls.gov/pub/time.series/pr/pr.duration,2026-09-03T08:30:00.000Z,176,null,null,null,null,null,null,null,null,null,null,new
pr.footnote,https://download.bls.gov/pub/time.series/pr/pr.footnote,2026-09-03T08:30:00.000Z,40,null,null,null,null,null,null,null,null,null,null,new
pr.measure,https://download.bls.gov/pub/time.series/pr/pr.measure,2026-09-03T08:30:00.000Z,745,null,null,null,null,null,null,null,null,null,null,new
pr.period,https://download.bls.gov/pub/time.series/pr/pr.period,1994-01-07T15:53:00.000Z,146,null,null,null,null,null,null,null,null,null,null,new
pr.seasonal,https://download.bls.gov/pub/time.series/pr/pr.seasonal,2011-11-18T16:05:00.000Z,79,null,null,null,null,null,null,null,null,null,null,new
pr.sector,https://download.bls.gov/pub/time.series/pr/pr.sector,2026-09-03T08:30:00.000Z,263,null,null,null,null,null,null,null,null,null,null,new


##### Step 12 — Change summary

In [0]:
display(
    df_file_changes
    .groupBy(
        "change_type"
    )
    .count()
)

change_type,count
new,12


##### Step 13 — Main processing loop

In [0]:
# ---------------------------------------------------------
# Step 13 - Main processing loop
# ---------------------------------------------------------

current_dt = datetime.now(
    timezone.utc
)

manifest_updates = []

for row in df_file_changes.collect():

    source_object = row["source_object"]
    change_type = row["change_type"]

    print(
        source_object,
        "->",
        change_type
    )

    # -----------------------------------------------------
    # 1. Already removed
    # -----------------------------------------------------

    if change_type == "already_removed":
        continue


    # -----------------------------------------------------
    # 2. Removed
    # -----------------------------------------------------

    if change_type == "removed":

        manifest_updates.append(
            build_manifest_record(
                source_object=source_object,
                source_url=row["tgt_source_url"],
                source_modified_dt=row["tgt_source_modified_dt"],
                source_size=row["tgt_source_size"],
                content_hash=row["tgt_content_hash"],
                landing_dir=row["tgt_landing_dir"],
                volume_path=row["tgt_volume_path"],
                status="removed",
                first_seen_dt=row["tgt_first_seen_dt"],
                last_seen_dt=row["tgt_last_seen_dt"],
                ingested_dt=row["tgt_ingested_dt"]
            )
        )

        continue


    # -----------------------------------------------------
    # 3. Unchanged
    # -----------------------------------------------------

    if change_type == "unchanged":

        manifest_updates.append(
            build_manifest_record(
                source_object=source_object,
                source_url=row["src_source_url"],
                source_modified_dt=row["src_source_modified_dt"],
                source_size=row["src_source_size"],
                content_hash=row["tgt_content_hash"],
                landing_dir=row["tgt_landing_dir"],
                volume_path=row["tgt_volume_path"],
                status="active",
                first_seen_dt=row["tgt_first_seen_dt"],
                last_seen_dt=current_dt,
                ingested_dt=row["tgt_ingested_dt"]
            )
        )

        continue


    # -----------------------------------------------------
    # 4. New / Changed Candidate / Reactivated
    # -----------------------------------------------------

    if change_type in [
        "new",
        "changed_candidate",
        "reactivated"
    ]:

        (
            content,
            actual_size,
            new_content_hash
        ) = download_and_hash(
            bls_session,
            row["src_source_url"],
            row["src_source_size"]
        )


        # -------------------------------------------------
        # 4A. Metadata changed but content is identical
        # -------------------------------------------------

        if (
            change_type == "changed_candidate"
            and
            new_content_hash == row["tgt_content_hash"]
        ):

            manifest_updates.append(
                build_manifest_record(
                    source_object=source_object,
                    source_url=row["src_source_url"],
                    source_modified_dt=row["src_source_modified_dt"],
                    source_size=actual_size,
                    content_hash=row["tgt_content_hash"],
                    landing_dir=row["tgt_landing_dir"],
                    volume_path=row["tgt_volume_path"],
                    status="active",
                    first_seen_dt=row["tgt_first_seen_dt"],
                    last_seen_dt=current_dt,
                    ingested_dt=row["tgt_ingested_dt"]
                )
            )

            continue


        # -------------------------------------------------
        # 4B. Actual new / changed / reactivated content
        # -------------------------------------------------

        landing_dir, volume_path = write_raw_version(
            bls_volume_path,
            source_object,
            content,
            current_dt
        )


        # -------------------------------------------------
        # Determine first_seen_dt
        # -------------------------------------------------

        if change_type == "new":

            first_seen_dt = (
                current_dt
            )

        else:

            first_seen_dt = (
                row["tgt_first_seen_dt"]
            )


        # -------------------------------------------------
        # Build manifest record
        # -------------------------------------------------

        manifest_updates.append(
            build_manifest_record(
                source_object=source_object,
                source_url=row["src_source_url"],
                source_modified_dt=row["src_source_modified_dt"],
                source_size=actual_size,
                content_hash=new_content_hash,
                landing_dir=landing_dir,
                volume_path=volume_path,
                status="active",
                first_seen_dt=first_seen_dt,
                last_seen_dt=current_dt,
                ingested_dt=current_dt
            )
        )
     

pr.class -> new
pr.contacts -> new
pr.data.0.Current -> new
pr.data.1.AllData -> new
pr.duration -> new
pr.footnote -> new
pr.measure -> new
pr.period -> new
pr.seasonal -> new
pr.sector -> new
pr.series -> new
pr.txt -> new


##### Step 14 — Build manifest update DataFrame

In [0]:
manifest_schema = (
    spark.table(
        manifest_table
    ).schema
)

if manifest_updates:

    df_manifest_updates = (
        spark.createDataFrame(
            manifest_updates,
            schema=manifest_schema
        )
    )

else:

    df_manifest_updates = (
        spark.createDataFrame(
            [],
            schema=manifest_schema
        )
    )


display(
    df_manifest_updates
)

source_name,source_object,source_url,source_modified_dt,source_size,content_hash,volume_path,status,first_seen_dt,last_seen_dt,ingested_dt,landing_dir
bls,pr.class,https://download.bls.gov/pub/time.series/pr/pr.class,2026-09-03T08:30:00.000Z,102,ed5e7641b5b990460f02a3189b22b48535abcdf3eb505ebfe820ef4c7a288bc9,/Volumes/rearc/bronze/raw/bls/inbound/pr.class/v=20260910T091923601096/pr.class,active,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,/Volumes/rearc/bronze/raw/bls/inbound/pr.class
bls,pr.contacts,https://download.bls.gov/pub/time.series/pr/pr.contacts,2022-09-13T16:52:00.000Z,562,6dfd5df62764766e9afe10b2de7ae8e1b8d372e44c2809f5bf292ee788b2d30a,/Volumes/rearc/bronze/raw/bls/inbound/pr.contacts/v=20260910T091923601096/pr.contacts,active,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,/Volumes/rearc/bronze/raw/bls/inbound/pr.contacts
bls,pr.data.0.Current,https://download.bls.gov/pub/time.series/pr/pr.data.0.Current,2026-09-03T08:30:00.000Z,1615931,5d43083f802cf1745778734529256d4ebd3a4a11ae44a36463837149994e46b6,/Volumes/rearc/bronze/raw/bls/inbound/pr.data.0.Current/v=20260910T091923601096/pr.data.0.Current,active,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,/Volumes/rearc/bronze/raw/bls/inbound/pr.data.0.Current
bls,pr.data.1.AllData,https://download.bls.gov/pub/time.series/pr/pr.data.1.AllData,2026-09-03T08:30:00.000Z,3239525,bf93b8a24a3c2e7e51d0dfe96536c20566ea3b3a05c7a45bb08c9ac123829d89,/Volumes/rearc/bronze/raw/bls/inbound/pr.data.1.AllData/v=20260910T091923601096/pr.data.1.AllData,active,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,/Volumes/rearc/bronze/raw/bls/inbound/pr.data.1.AllData
bls,pr.duration,https://download.bls.gov/pub/time.series/pr/pr.duration,2026-09-03T08:30:00.000Z,176,d1132a2ac0a27c12a1bb82538be3cb5ba1dfaa3eb6951feb8caba5d99202e97f,/Volumes/rearc/bronze/raw/bls/inbound/pr.duration/v=20260910T091923601096/pr.duration,active,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,/Volumes/rearc/bronze/raw/bls/inbound/pr.duration
bls,pr.footnote,https://download.bls.gov/pub/time.series/pr/pr.footnote,2026-09-03T08:30:00.000Z,40,29f9274a6e9da7dfc5e2258144d9edbaed34cbe02d852c82aab20c2e8ac3148b,/Volumes/rearc/bronze/raw/bls/inbound/pr.footnote/v=20260910T091923601096/pr.footnote,active,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,/Volumes/rearc/bronze/raw/bls/inbound/pr.footnote
bls,pr.measure,https://download.bls.gov/pub/time.series/pr/pr.measure,2026-09-03T08:30:00.000Z,745,459c07d625b60430028569cf37f20c4b618f3de0ece5756ef6fa034cbeb594c6,/Volumes/rearc/bronze/raw/bls/inbound/pr.measure/v=20260910T091923601096/pr.measure,active,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,/Volumes/rearc/bronze/raw/bls/inbound/pr.measure
bls,pr.period,https://download.bls.gov/pub/time.series/pr/pr.period,1994-01-07T15:53:00.000Z,146,2a73359647de19a1fc85b2db5002d7efdda291005a501bfe432749c60bce0b2c,/Volumes/rearc/bronze/raw/bls/inbound/pr.period/v=20260910T091923601096/pr.period,active,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,/Volumes/rearc/bronze/raw/bls/inbound/pr.period
bls,pr.seasonal,https://download.bls.gov/pub/time.series/pr/pr.seasonal,2011-11-18T16:05:00.000Z,79,5ebe468ab3cbcfdd6b19b81f34af9281dd4728e891d78192f35de3a5bd9d7b24,/Volumes/rearc/bronze/raw/bls/inbound/pr.seasonal/v=20260910T091923601096/pr.seasonal,active,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,/Volumes/rearc/bronze/raw/bls/inbound/pr.seasonal
bls,pr.sector,https://download.bls.gov/pub/time.series/pr/pr.sector,2026-09-03T08:30:00.000Z,263,1eaef7fb47ac5accc468ff39c4e8916f575a644a009ee66ef8dbd188fe54e369,/Volumes/rearc/bronze/raw/bls/inbound/pr.sector/v=20260910T091923601096/pr.sector,active,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,/Volumes/rearc/bronze/raw/bls/i

##### Step 15 — Validate duplicate merge keys

In [0]:
df_manifest_duplicates = (
    df_manifest_updates
    .groupBy(
        "source_name",
        "source_object"
    )
    .count()
    .filter(
        f.col("count") > 1
    )
)

if df_manifest_duplicates.count() > 0:

    display(
        df_manifest_duplicates
    )

    raise ValueError(
        "duplicate manifest merge keys detected"
    )

##### Step 16 — MERGE manifest

In [0]:
manifest_delta = (
    DeltaTable.forName(
        spark,
        manifest_table
    )
)

if df_manifest_updates.count() > 0:

    (
        manifest_delta
        .alias("tgt")

        .merge(
            df_manifest_updates
            .alias("src"),

            """
            tgt.source_name = src.source_name
            AND
            tgt.source_object = src.source_object
            """
        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()
    )

print(
    "Manifest MERGE completed"
)

Manifest MERGE completed


##### Step 17 — Final manifest validation

In [0]:
display(
    spark.table(
        manifest_table
    )
    .filter(
        f.col("source_name")
        == "bls"
    )
    .select(
        "source_object",
        "status",
        "source_modified_dt",
        "source_size",
        "content_hash",
        "landing_dir",
        "volume_path",
        "first_seen_dt",
        "last_seen_dt",
        "ingested_dt"
    )
    .orderBy(
        "source_object"
    )
)

source_object,status,source_modified_dt,source_size,content_hash,landing_dir,volume_path,first_seen_dt,last_seen_dt,ingested_dt
pr.class,active,2026-09-03T08:30:00.000Z,102,ed5e7641b5b990460f02a3189b22b48535abcdf3eb505ebfe820ef4c7a288bc9,/Volumes/rearc/bronze/raw/bls/inbound/pr.class,/Volumes/rearc/bronze/raw/bls/inbound/pr.class/v=20260910T091923601096/pr.class,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z
pr.contacts,active,2022-09-13T16:52:00.000Z,562,6dfd5df62764766e9afe10b2de7ae8e1b8d372e44c2809f5bf292ee788b2d30a,/Volumes/rearc/bronze/raw/bls/inbound/pr.contacts,/Volumes/rearc/bronze/raw/bls/inbound/pr.contacts/v=20260910T091923601096/pr.contacts,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z
pr.data.0.Current,active,2026-09-03T08:30:00.000Z,1615931,5d43083f802cf1745778734529256d4ebd3a4a11ae44a36463837149994e46b6,/Volumes/rearc/bronze/raw/bls/inbound/pr.data.0.Current,/Volumes/rearc/bronze/raw/bls/inbound/pr.data.0.Current/v=20260910T091923601096/pr.data.0.Current,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z
pr.data.1.AllData,active,2026-09-03T08:30:00.000Z,3239525,bf93b8a24a3c2e7e51d0dfe96536c20566ea3b3a05c7a45bb08c9ac123829d89,/Volumes/rearc/bronze/raw/bls/inbound/pr.data.1.AllData,/Volumes/rearc/bronze/raw/bls/inbound/pr.data.1.AllData/v=20260910T091923601096/pr.data.1.AllData,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z
pr.duration,active,2026-09-03T08:30:00.000Z,176,d1132a2ac0a27c12a1bb82538be3cb5ba1dfaa3eb6951feb8caba5d99202e97f,/Volumes/rearc/bronze/raw/bls/inbound/pr.duration,/Volumes/rearc/bronze/raw/bls/inbound/pr.duration/v=20260910T091923601096/pr.duration,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z
pr.footnote,active,2026-09-03T08:30:00.000Z,40,29f9274a6e9da7dfc5e2258144d9edbaed34cbe02d852c82aab20c2e8ac3148b,/Volumes/rearc/bronze/raw/bls/inbound/pr.footnote,/Volumes/rearc/bronze/raw/bls/inbound/pr.footnote/v=20260910T091923601096/pr.footnote,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z
pr.measure,active,2026-09-03T08:30:00.000Z,745,459c07d625b60430028569cf37f20c4b618f3de0ece5756ef6fa034cbeb594c6,/Volumes/rearc/bronze/raw/bls/inbound/pr.measure,/Volumes/rearc/bronze/raw/bls/inbound/pr.measure/v=20260910T091923601096/pr.measure,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z
pr.period,active,1994-01-07T15:53:00.000Z,146,2a73359647de19a1fc85b2db5002d7efdda291005a501bfe432749c60bce0b2c,/Volumes/rearc/bronze/raw/bls/inbound/pr.period,/Volumes/rearc/bronze/raw/bls/inbound/pr.period/v=20260910T091923601096/pr.period,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z
pr.seasonal,active,2011-11-18T16:05:00.000Z,79,5ebe468ab3cbcfdd6b19b81f34af9281dd4728e891d78192f35de3a5bd9d7b24,/Volumes/rearc/bronze/raw/bls/inbound/pr.seasonal,/Volumes/rearc/bronze/raw/bls/inbound/pr.seasonal/v=20260910T091923601096/pr.seasonal,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z
pr.sector,active,2026-09-03T08:30:00.000Z,263,1eaef7fb47ac5accc468ff39c4e8916f575a644a009ee66ef8dbd188fe54e369,/Volumes/rearc/bronze/raw/bls/inbound/pr.sector,/Volumes/rearc/bronze/raw/bls/inbound/pr.sector/v=20260910T091923601096/pr.sector,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z,2026-09-10T09:19:23.601Z


##### Step 18 — Check statuses

In [0]:
display(
    spark.table(
        manifest_table
    )
    .filter(
        f.col("source_name")
        == "bls"
    )
    .groupBy(
        "status"
    )
    .count()
)

status,count
active,12
